# Load and Evaluate ClapNQ via Unitxt  

For more details regarding Unitxt see here: https://unitxt.readthedocs.io/en/latest/docs/introduction.html

In [10]:
from transformers import pipeline
from datasets import load_dataset

dataset = load_dataset(
    "unitxt/data",
    "card=cards.rag.response_generation.clapnq"
    ",template_card_index=answer_based_on_context"
    ",demos_pool_size=10,num_demos=2",
    trust_remote_code=True,
)


Repo card metadata block was not found. Setting CardData to empty.


Loading with huggingface unitxt copy...
'prediction_type' was not set in Task. It is used to check the output of template post processors is compatible with the expected input of the metrics. Setting `prediction_type` to 'Any' (no checking is done). In future version of unitxt this will raise an exception.
LoadHF sets 'data_classification_policy' to ['public'] by default when loading from Huggingface hub.
To use a different value or remove this message, explicitly set the `data_classification_policy` attribute of the loader.



Generating test split: 600 examples [00:01, 529.08 examples/s]
Generating demos_pool split: 10 examples [00:00, 31.80 examples/s]
Generating train split: 3735 examples [00:04, 747.69 examples/s]


In [11]:
# dataset info
dataset

DatasetDict({
    test: Dataset({
        features: ['metrics', 'data_classification_policy', 'target', 'references', 'source', 'task_data', 'group', 'postprocessors'],
        num_rows: 600
    })
    demos_pool: Dataset({
        features: ['metrics', 'data_classification_policy', 'target', 'references', 'source', 'task_data', 'group', 'postprocessors'],
        num_rows: 10
    })
    train: Dataset({
        features: ['metrics', 'data_classification_policy', 'target', 'references', 'source', 'task_data', 'group', 'postprocessors'],
        num_rows: 3735
    })
})

In [13]:
# get small demos_pool and print first example
subset = dataset['demos_pool']
subset[0]

{'metrics': ['metrics.rag.response_generation.correctness.token_overlap',
  'metrics.rag.response_generation.faithfullness.token_overlap',
  'metrics.rag.response_generation.correctness.bert_score.deberta_large_mnli'],
 'data_classification_policy': ['public'],
 'target': "`` Love the One You 're With '' is a song by folk rocker Stephen Stills . David Crosby and Graham Nash , Stills ' fellow members of Crosby , Stills & Nash , provide background vocals on the song . The song was also covered by a number of artists , including The Isley Brothers , Bucks Fizz , and Luther Vandross .",
 'references': ["`` Love the One You 're With '' is a song by folk rocker Stephen Stills . David Crosby and Graham Nash , Stills ' fellow members of Crosby , Stills & Nash , provide background vocals on the song . The song was also covered by a number of artists , including The Isley Brothers , Bucks Fizz , and Luther Vandross ."],
 'source': "Answer the question, basing your answer on the context\nContext:

In [18]:
# These lines can be replaces by any inference engine, that receives the model_input strings
# and returns models predictions as string.
model_inputs = subset['source']
model = pipeline(model='google/flan-t5-base')
predictions = [output['generated_text'] for output in model(model_inputs,max_new_tokens=100)]

Token indices sequence length is longer than the specified maximum sequence length for this model (883 > 512). Running this sequence through the model will result in indexing errors


In [19]:
# Evaluate with Unitxt: https://unitxt.readthedocs.io/en/latest/docs/evaluating_datasets.html
import evaluate

metric = evaluate.load('unitxt/metric')
dataset_with_scores = metric.compute(predictions=predictions,references=subset)

[print(item) for item in dataset_with_scores[0]['score']['global'].items()]

('f1', 0.23942552655852953)
('score', 0.23942552655852953)
('score_name', 'f1')
('precision', 0.8140958605664489)
('recall', 0.17858103935276026)
('precision_ci_low', 0.5887303309108072)
('precision_ci_high', 0.9470588235294117)
('recall_ci_low', 0.08254178892774582)
('recall_ci_high', 0.3355698520010427)
('f1_ci_low', 0.13689296891700475)
('f1_ci_high', 0.42120326550052306)
('score_ci_low', 0.13689296891700475)
('score_ci_high', 0.42120326550052306)
('correctness_f1_bert_score.deberta_large_mnli', 0.5282835394144059)
('correctness_recall_bert_score.deberta_large_mnli', 0.44590863287448884)
('correctness_precision_bert_score.deberta_large_mnli', 0.6702001333236695)
('faithfullness_f1_token_overlap', 0.12805502573304695)
('faithfullness_recall_token_overlap', 0.07553647981375657)
('faithfullness_precision_token_overlap', 0.9444444444444444)
('correctness_f1_token_overlap', 0.23942552655852953)
('correctness_recall_token_overlap', 0.17858103935276026)
('correctness_precision_token_overla

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]